In [2]:
import numpy as np

# Parameters — pick any reasonable values
n = 6
alpha = 0.5
tilde_c_inv = 0.3   # set to 0.0 for the unregularized case
i_test, j_test = 2, 4

# Derived quantities
alpha_prime = (1 - alpha) / 2
lam = np.arccosh((1 + tilde_c_inv) / (1 - alpha))

# Build B = K~ + tilde_c^{-1} I, an (n-1) x (n-1) tridiagonal matrix
# K~ has 1 on diagonal, -(1-alpha)/2 on immediate off-diagonals, 0 elsewhere
N = n - 1
B = np.zeros((N, N))
for k in range(N):
    B[k, k] = 1 + tilde_c_inv
for k in range(N - 1):
    B[k, k+1] = -(1 - alpha) / 2
    B[k+1, k] = -(1 - alpha) / 2

B_inv = np.linalg.inv(B)

# Pad B_inv with zeros for index 0 and index n (i.e., outside [1, n-1])
def Binv(i, j):
    """1-indexed access; return 0 if i or j is outside [1, n-1]."""
    if i < 1 or j < 1 or i > n - 1 or j > n - 1:
        return 0.0
    return B_inv[i - 1, j - 1]

# Direct computation of D_{ij} from definition
def D_direct(i, j):
    return alpha_prime * (Binv(i - 1, j) + Binv(i, j - 1) - Binv(i, j) - Binv(i - 1, j - 1))

# Closed-form Case 1 (i <= j-1):  +2 * cosh((i-1/2)lam) * cosh((n-j+1/2)lam) * tanh(lam/2) / sinh(n*lam)
def D_closed_case1(i, j):
    return (2 * np.cosh((i - 0.5) * lam) * np.cosh((n - j + 0.5) * lam)
            * np.tanh(lam / 2) / np.sinh(n * lam))

# Closed-form Case 1 with NEGATIVE sign — for comparison
def D_closed_case1_neg(i, j):
    return -D_closed_case1(i, j)

# Test for i <= j-1
print(f"n={n}, alpha={alpha}, tilde_c_inv={tilde_c_inv}, lambda={lam:.4f}")
print(f"\nTesting D_{{i,j}} for i <= j-1 cases:\n")
print(f"{'(i,j)':<8} {'Direct':<14} {'Closed (+2)':<14} {'Closed (-2)':<14} {'Match':<10}")
print("-" * 60)

for i in range(1, n):
    for j in range(i + 1, n + 1):
        d_direct = D_direct(i, j)
        d_pos = D_closed_case1(i, j)
        d_neg = D_closed_case1_neg(i, j)
        match_pos = "✓ +2" if np.isclose(d_direct, d_pos) else ""
        match_neg = "✓ -2" if np.isclose(d_direct, d_neg) else ""
        match = match_pos + match_neg
        print(f"({i},{j}){'':<3} {d_direct:<14.6f} {d_pos:<14.6f} {d_neg:<14.6f} {match}")

# Also test the diagonal case D_{ii}
print(f"\nTesting D_{{i,i}} (diagonal case):\n")
print(f"{'i':<5} {'Direct':<14} {'Closed (-2(1-...))':<20} {'Match':<10}")
print("-" * 55)

def D_closed_case3(i):
    """Tentative closed form for i = j: -2(1 - cosh()cosh()tanh / sinh(nlam))"""
    return -2 * (1 - np.cosh((i - 0.5) * lam) * np.cosh((n - i + 0.5) * lam)
                 * np.tanh(lam / 2) / np.sinh(n * lam))

for i in range(1, n):
    d_direct = D_direct(i, i)
    d_closed = D_closed_case3(i)
    match = "✓" if np.isclose(d_direct, d_closed) else "✗"
    print(f"{i:<5} {d_direct:<14.6f} {d_closed:<20.6f} {match}")

n=6, alpha=0.5, tilde_c_inv=0.3, lambda=1.6094

Testing D_{i,j} for i <= j-1 cases:

(i,j)    Direct         Closed (+2)    Closed (-2)    Match     
------------------------------------------------------------
(1,2)    0.160000       0.160000       -0.160000      ✓ +2
(1,3)    0.032000       0.032000       -0.032000      ✓ +2
(1,4)    0.006402       0.006402       -0.006402      ✓ +2
(1,5)    0.001290       0.001290       -0.001290      ✓ +2
(1,6)    0.000307       0.000307       -0.000307      ✓ +2
(2,3)    0.134402       0.134402       -0.134402      ✓ +2
(2,4)    0.026889       0.026889       -0.026889      ✓ +2
(2,5)    0.005419       0.005419       -0.005419      ✓ +2
(2,6)    0.001290       0.001290       -0.001290      ✓ +2
(3,4)    0.133419       0.133419       -0.133419      ✓ +2
(3,5)    0.026889       0.026889       -0.026889      ✓ +2
(3,6)    0.006402       0.006402       -0.006402      ✓ +2
(4,5)    0.134402       0.134402       -0.134402      ✓ +2
(4,6)    0.032000     